# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: diagrams, shapes, points, vectors, segments and adjacency specs, each a reference object owned through `Logical.Owns`, each playing its part through a relation of its own, with no coordinates; and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-237.md), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import { Builders, JSON as Json, Paths, Validators } from "@mbse/schemas/Framework";

import { Logical } from "../src/Framework/index.js";
import { assert, text } from "./support.js";

const store: any = new Builders.Store();
store.transaction(Logical.install);
const L = store.Logical;
const directory = store.singleton("Logical.Diagrams");
const problems = (schema: any, root: any): string[] => Validators.Validate(store).Reachable(schema, root);

/** The entry by which `owner` owns what it builds, as `name`. */
function owned(owner: any, name: string): (a: any) => any {
  return (a: any) => a.insert((e: any) => e.name(name).owner(owner));
}

function diagram(name: string): any {
  return L.OfDiagram((d: any) => d.owner(owned(directory, name)).directory((a: any) => a.insert((e: any) => e.directory(directory))));
}

/** A shape owned by `container`, a diagram or a shape, and in it. */
function shape(container: any, name: string): any {
  const role = container.as_schema().name === "Logical.OfDiagram"
    ? (s: any) => s.diagram((a: any) => a.insert((e: any) => e.diagram(container)))
    : (s: any) => s.container((a: any) => a.insert((e: any) => e.container(container)));
  return L.OfShape((s: any) => role(s.owner(owned(container, name))));
}

/** A point owned by the shape `of`, as its origin, a border or an extent point. */
function point(of: any, name: string, role: string): any {
  const link = ({ origin: "origin_of", border: "border_of", extent: "extent_of" } as Record<string, string>)[role] as string;
  return L.OfPoint((p: any) => p.owner(owned(of, name))[link]((a: any) => a.insert((e: any) => e.shape(of))));
}

function vector(of: any, name: string, start: any, end: any, degrees: number | null = null, length = 1.0): any {
  return L.OfVector((v: any) => {
    v = v.owner(owned(of, name)).start((a: any) => a.insert((e: any) => e.point(start)))
      .end((a: any) => a.insert((e: any) => e.point(end))).length(length);
    return degrees === null ? v : v.rotation((r: any) => r.degrees(degrees));
  });
}

function raises(message: string, block: () => unknown): void {
  try {
    store.transaction(block);
  } catch (error) {
    assert(error instanceof Error && error.name === "ValueError", String(error));
    assert(error.message === message, error.message);
    return;
  }
  throw new Error(`AssertionError: expected ValueError: ${message}`);
}

const path = (x: any): string => Json.ToJSON(store)(store.Paths.OfRoot.as_schema(), Paths.of(x));
const names = (owner: any): string[] => [...owner.owns].map((e: any) => e.name.get());

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
const NAMES = ["Logical.Owns", "Logical.OfRotation", "Logical.DiagramDirectory", "Logical.InDiagrams", "Logical.OfDiagram", "Logical.InDiagram", "Logical.OfShape", "Logical.Containment", "Logical.OfPoint", "Logical.Origin", "Logical.Borders", "Logical.Extents", "Logical.OfVector", "Logical.VectorStart", "Logical.VectorEnd", "Logical.OfSegment", "Logical.SegmentEnds", "Logical.OfAdjacency", "Logical.AdjacencyFrom", "Logical.AdjacencyTo"];
const schemas = NAMES.map((name) => store.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
store.transaction(Logical.install);
assert(NAMES.every((name, i) => store.registered(name) === schemas[i]) && directory === store.Instances.Logical.Diagrams);

<a id="LOG-02"></a>
## LOG-02 · Everything is owned through Logical.Owns, by a name unique among its owner's, its path read back through it

In [ ]:
let overview: any, cpu: any, alu: any, o: any, ne: any, e: any;
store.transaction(() => {
  overview = diagram("overview");
  cpu = shape(overview, "cpu");
  alu = shape(cpu, "alu");
  [o, ne, e] = [point(cpu, "origin", "origin"), point(cpu, "ne", "border"), point(cpu, "e", "extent")];
  vector(cpu, "to ne", o, ne, 45, 2);
  vector(cpu, "to e", o, e, 90);
  point(alu, "origin", "origin");
});
assert(text([names(directory), names(overview), names(cpu), names(alu)]) === "[[\"overview\"],[\"cpu\"],[\"alu\",\"origin\",\"ne\",\"e\",\"to ne\",\"to e\"],[\"origin\"]]");
assert(text([[overview, directory], [cpu, overview], [alu, cpu], [o, cpu]].map(([x, owner]) => Logical.owner_of(x) === owner)) === "[true,true,true,true]");
assert(text(path(alu.owns({ name: "origin" }))) === "\"{\\\"singleton\\\": \\\"Logical.Diagrams\\\", \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"overview\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"cpu\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"alu\\\"}, \\\"next\\\": {\\\"entry\\\": {\\\"i\\\": \\\"owns\\\", \\\"o\\\": \\\"owner\\\", \\\"properties\\\": {\\\"name\\\": \\\"origin\\\"}, \\\"next\\\": {\\\"target\\\": {}}}}}}}}}}}\"");
assert(text([Logical.owner_of(directory), Logical.name_of(directory), Logical.name_of(cpu)]) === "[null,null,\"cpu\"]");
assert(text(problems(L.OfDiagram, overview)) === "[]");

<a id="LOG-03"></a>
## LOG-03 · A shape's roles: in its diagram or its container, one origin, and border and extent points; refused without an owner

In [ ]:
assert(text([[...overview.shapes].map((x: any) => x.shape === cpu), [...cpu.children].map((x: any) => x.child === alu), [...alu.container].map((x: any) => x.container === cpu)]) === "[[true],[true],[true]]");
assert(text([[...cpu.origin].map((x: any) => x.point === o), [...cpu.borders].map((x: any) => x.point === ne), [...cpu.extents].map((x: any) => x.point === e)]) === "[[true],[true],[true]]");
store.transaction(() => point(cpu, "second", "origin"));
assert(text(problems(L.OfShape, cpu)) === "[\"unique(point) violated: entries agreeing on ['shape'] differ on ['point']\"]");
store.transaction(() => L.OfShape(cpu, (s: any) => s.owns((a: any) => a.remove((e: any) => e.name("second")))));
assert(text([problems(L.OfShape, cpu), names(cpu)]) === "[[],[\"alu\",\"origin\",\"ne\",\"e\",\"to ne\",\"to e\"]]");
raises("a 'Logical.OfPoint' needs an owner through 'owner': give it one, since no singleton can be",
  () => L.OfPoint((p: any) => p));

<a id="LOG-04"></a>
## LOG-04 · A vector runs from one start to one end, by a rotation and a length

In [ ]:
const v = cpu.owns({ name: "to ne" });
assert(text([Logical.ends(v)[0] === o, Logical.ends(v)[1] === ne, v.rotation.degrees.get(), v.length.get()]) === "[true,true,45,2]");
assert(text([[...ne.incoming].map((x: any) => x.vector === v), [...o.outgoing].length]) === "[[true],2]");
store.transaction(() => L.OfVector(v, (x: any) => x.end((a: any) => a.insert((e: any) => e.point(o)))));
assert(text(problems(L.OfVector, v)) === "[\"unique(point) violated: entries agreeing on ['vector'] differ on ['point']\"]");
store.transaction(() => L.OfVector(v, (x: any) => x.end((a: any) => a.remove((e: any) => e.point(o)))));
assert(text(problems(L.OfVector, v)) === "[]");

<a id="LOG-05"></a>
## LOG-05 · A segment joins two points, with no length or rotation; an adjacency spec runs from one shape to another

In [ ]:
let seg: any, fpu: any, spec: any;
store.transaction(() => {
  seg = L.OfSegment((s: any) => s.owner(owned(cpu, "edge")).ends((a: any) => a.insert((x: any) => x.point(ne)).insert(
    (x: any) => x.point(e))));
  fpu = shape(cpu, "fpu");
  spec = L.OfAdjacency((s: any) => s.owner(owned(alu, "to fpu 90")).from_shape((a: any) => a.insert((x: any) => x.shape(alu)))
    .to_shape((a: any) => a.insert((x: any) => x.shape(fpu))).rotation((r: any) => r.degrees(90)));
});
assert(text([[...seg.ends].map((x: any, i: number) => x.point === [ne, e][i]), [...e.segments].map((x: any) => x.segment === seg)]) === "[[true,true],[true]]");
assert(text([[...alu.adjacent_to].map((x: any) => x.adjacency === spec), [...fpu.adjacent_from].map((x: any) => x.adjacency === spec), spec.rotation.degrees.get(), names(alu)]) === "[[true],[true],90,[\"origin\",\"to fpu 90\"]]");
assert(text(problems(L.OfDiagram, overview)) === "[]");

<a id="LOG-06"></a>
## LOG-06 · A compass: a shape's and its origin's north, a border or extent point's its vector's rotation

In [ ]:
let sw: any, w: any;
store.transaction(() => {
  sw = point(cpu, "sw", "extent");
  vector(cpu, "to sw", o, sw, 585.5);
  w = point(cpu, "w", "border");
  vector(cpu, "to w", o, w, -90);
});
assert(text([cpu, o, alu, ne, e, sw, w].map((x) => Logical.compass(x))) === "[0,0,0,45,90,225.5,270]");

<a id="LOG-07"></a>
## LOG-07 · A compass the rules leave undefined is refused

In [ ]:
let bare: any, x: any, q: any, s: any, t: any;
store.transaction(() => {
  bare = shape(cpu, "bare");
  x = point(bare, "x", "border");
  q = point(fpu, "q", "border");
  point(fpu, "origin", "origin");
  s = point(cpu, "s", "border");
  vector(cpu, "to s", o, s);
  t = point(cpu, "t", "border");
  vector(cpu, "to t 1", o, t, 1);
  vector(cpu, "to t 2", o, t, 2);
});
raises("a shape without an origin has no compass", () => Logical.compass(bare));
raises("a shape without an origin has no compass", () => Logical.compass(x));
raises("a point with 0 vectors from its shape's origin, not one, has no compass", () => Logical.compass(q));
raises("a point with 2 vectors from its shape's origin, not one, has no compass", () => Logical.compass(t));
raises("a point whose vector from its shape's origin has no rotation has no compass", () => Logical.compass(s));